In [ ]:
# Imports & load aggregated dataset

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})

DATA_JSON = "../data/quotes_grouped.json"  # produced by src/parse_quotes.py
FIG_DIR = Path("figures_eda"); FIG_DIR.mkdir(parents=True, exist_ok=True)

df_all = pd.read_json(DATA_JSON)
df_price = df_all[df_all["proposed_price_accepted"].notnull()].copy()

print("df_all:", df_all.shape, "| df_price:", df_price.shape)
df_all.head()

In [ ]:
# Missing values & quick checks

missing = df_all.isnull().sum().sort_values(ascending=False)
missing[missing > 0].head(15)

In [ ]:
# Top languages & document types

import seaborn as sns
import matplotlib.pyplot as plt

# Top source languages
plt.figure(figsize=(10, 8))
order_src = df_all["source_language"].value_counts().head(25).index
sns.countplot(data=df_all[df_all["source_language"].isin(order_src)],
              y="source_language", order=order_src)
plt.title("Most common source languages")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_bar_source_languages.png", dpi=180); plt.show()

# Top target languages
plt.figure(figsize=(10, 8))
order_tgt = df_all["target_language"].value_counts().head(25).index
sns.countplot(data=df_all[df_all["target_language"].isin(order_tgt)],
              y="target_language", order=order_tgt)
plt.title("Most common target languages")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_bar_target_languages.png", dpi=180); plt.show()

# Top document types
plt.figure(figsize=(10, 6))
top_docs = df_all["document_type"].value_counts().head(15)
sns.barplot(x=top_docs.values, y=top_docs.index, orient="h")
plt.title("Most common document types")
plt.xlabel("Number of occurrences")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_bar_document_types.png", dpi=180); plt.show()

In [ ]:
# Volume (pages), acceptation

# Pages (cap at 70)
plt.figure(figsize=(8, 4))
sns.histplot(df_all[df_all["number_of_unit"] < 70]["number_of_unit"], bins=30, kde=True)
plt.title("Distribution of the number of pages")
plt.xlabel("number_of_unit")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_hist_pages.png", dpi=180); plt.show()

# Accepted vs rejected
plt.figure(figsize=(6, 4))
sns.countplot(x="price_accepted", data=df_all)
plt.title("Quotes accepted vs. rejected")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_bar_accepted_vs_rejected.png", dpi=180); plt.show()

In [ ]:
# Prix acceptés (attention devise)

# If currencies are heterogeneous, draw per-currency histograms
plt.figure(figsize=(8, 4))
sns.histplot(df_price[df_price["proposed_price_accepted"] < 600]["proposed_price_accepted"], bins=40, kde=True)
plt.title("Distribution of accepted prices (cap €600)")
plt.xlabel("Accepted price (original currency)")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_hist_accepted_prices_allcurr.png", dpi=180); plt.show()

# Per-currency hist (top 4 currencies)
top_cur = df_price["currency"].astype(str).value_counts().head(4).index.tolist()
g = sns.displot(
    data=df_price[df_price["currency"].astype(str).isin(top_cur) & (df_price["proposed_price_accepted"] < 600)],
    x="proposed_price_accepted", col="currency", col_wrap=2, bins=40, kde=True, height=3.2
)
g.set_titles("Currency: {col_name}")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_hist_accepted_prices_by_currency.png", dpi=180); plt.show()

In [ ]:
# Effets par catégories (doc, taille, deadline)

# Boxplot by document type
subset = df_price[df_price["proposed_price_accepted"] < 600].copy()
plt.figure(figsize=(14, 7))
order_doc = subset.groupby("document_type")["proposed_price_accepted"].median().sort_values().index[-20:]
sns.boxplot(data=subset[subset["document_type"].isin(order_doc)],
            x="document_type", y="proposed_price_accepted", order=order_doc)
plt.xticks(rotation=40, ha="right")
plt.title("Accepted price by document type (cap 600)")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_box_price_by_document_type.png", dpi=180); plt.show()

# Box/violin by size bins (pages)
bins = [0,1,2,3,5,8,12,20,35,60,1000]
labels = ["0-1","1-2","2-3","3-5","5-8","8-12","12-20","20-35","35-60","60+"]
sub = df_price.copy()
sub["size_bin"] = pd.cut(sub["number_of_unit"].fillna(0), bins=bins, labels=labels, right=False)
plt.figure(figsize=(14,6))
sns.boxplot(data=sub[sub["proposed_price_accepted"]<600], x="size_bin", y="proposed_price_accepted")
plt.title("Accepted price by size bin (pages, cap 600)")
plt.tight_layout(); plt.savefig(FIG_DIR/"eda_box_price_by_size_bin.png", dpi=180); plt.show()

# Deadline (if present)
if "requested_deadline" in df_price.columns:
    sub = df_price.copy()
    sub["requested_deadline"] = pd.to_numeric(sub["requested_deadline"], errors="coerce")
    sub = sub[sub["requested_deadline"].notnull()]
    if not sub.empty:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=sub[sub["proposed_price_accepted"]<600],
                    x="requested_deadline", y="proposed_price_accepted")
        plt.title("Accepted price by requested deadline (days)")
        plt.tight_layout(); plt.savefig(FIG_DIR/"eda_box_price_by_deadline.png", dpi=180); plt.show()

In [ ]:
# Time series (robust to missing/invalid timestamps)
import pandas as pd
import matplotlib.pyplot as plt

if "created_at" in df_all.columns:
    # Parse and drop rows with invalid timestamps
    dfts = df_all.copy()
    dfts["created_at"] = pd.to_datetime(dfts["created_at"], errors="coerce")
    dfts = dfts.dropna(subset=["created_at"])
    if not dfts.empty:
        ts = dfts.set_index("created_at").sort_index()

        # Monthly average of accepted price (all currencies) — filter to numeric
        s_price = pd.to_numeric(ts["proposed_price_accepted"], errors="coerce")
        monthly_mean = s_price.resample("MS").mean()

        # Monthly quote count
        monthly_cnt = ts["quote_no"].resample("MS").count()

        plt.figure(figsize=(10,4))
        monthly_mean.plot()
        plt.title("Monthly average accepted price (all currencies)")
        plt.tight_layout(); plt.savefig(FIG_DIR/"eda_ts_avg_price.png", dpi=180); plt.show()

        plt.figure(figsize=(10,4))
        monthly_cnt.plot()
        plt.title("Monthly quote count")
        plt.tight_layout(); plt.savefig(FIG_DIR/"eda_ts_quote_count.png", dpi=180); plt.show()
    else:
        print("No valid created_at timestamps in this snapshot — time series skipped.")
else:
    print("'created_at' column not present — time series skipped.")

In [ ]:
# Load secrets from .env and run the diagnostic
from dotenv import load_dotenv
load_dotenv("../.env")

HOST = os.getenv("HOST") or "YOUR_HOST"
PORT = int(os.getenv("PORT", "3306"))  # if you setup an SSH tunnel, set PORT=13306 and HOST=127.0.0.1
USER = os.getenv("DB_USER")
PWD  = os.getenv("PASSWORD")
DB   = os.getenv("DB_NAME")

engine = diagnose_mysql(HOST, PORT, USER, PWD, DB)

if engine is None:
    print("\n❌ Could not establish a connection. See logs above.")
else:
    print("\n✅ MySQL connection is working.")